# CLIP K-Fold CV — Config Selection

Selects best unfreezing config via 5-fold stratified CV.
See `status.md` for full plan.

In [1]:
import os, glob, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
import open_clip
from sklearn.model_selection import StratifiedKFold

print('Imports OK')

Imports OK


In [2]:
# ── Config ──────────────────────────────────────────────────
SEED             = 42
K_FOLDS          = 4    # 4 ensures proper stratification (class 42 has only 4 images)
NUM_CLASSES      = 100
BATCH_SIZE       = 64
NUM_WORKERS      = 0
PATIENCE_A       = 10   # early stopping patience — Phase A
PATIENCE_B       = 5    # early stopping patience — Phase B
MAX_EPOCHS_A     = 60   # ceiling; actual budget set by CV median
MAX_EPOCHS_B     = 30   # ceiling; actual budget set by CV median
LR_HEAD          = 1e-3
WEIGHT_DECAY     = 1e-4
USE_AUG          = False  # toggle: horizontal flip only (CLIP-safe); does NOT affect head_only

# CLIP_MODEL      = 'ViT-B-32'
# CLIP_PRETRAINED = 'openai'
CLIP_MODEL      = 'ViT-B-16'
CLIP_PRETRAINED = 'laion2b_s34b_b88k'

DATA_DIR  = './data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR  = os.path.join(DATA_DIR, 'test')
CKPT_DIR  = './checkpoints_kfold'
os.makedirs(CKPT_DIR, exist_ok=True)

# n_unfreeze = how many LAST blocks to unfreeze — resolved to indices at runtime
# so configs work correctly for any backbone depth (B/32=12 blocks, L/14=24 blocks)
# lr_backbone on head_only is unused but kept for uniform dict shape
CONFIGS = [
    {'name': 'head_only',      'n_unfreeze': 0, 'lr_backbone': 1e-6},
    {'name': 'unfreeze_2_1e6', 'n_unfreeze': 2, 'lr_backbone': 1e-6},
    {'name': 'unfreeze_2_1e5', 'n_unfreeze': 2, 'lr_backbone': 1e-5},
    {'name': 'unfreeze_4_1e5', 'n_unfreeze': 4, 'lr_backbone': 1e-5},
]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device(
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available() else 'cpu'
)
print('device:', device)

device: cuda


In [3]:
# ── Load CLIP + save original weights ───────────────────────
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAINED
)
clip_model = clip_model.to(device)
clip_model.eval()

# Derive embedding dim from model — never hardcode 512
EMBED_DIM = clip_model.visual.output_dim

# Save original backbone state — restored before each fold's Phase B
original_clip_state = {k: v.clone().cpu() for k, v in clip_model.state_dict().items()}

print(f'Model: {CLIP_MODEL} pretrained={CLIP_PRETRAINED}')
print(f'Embed dim: {EMBED_DIM}')
print(f'Total params: {sum(p.numel() for p in clip_model.parameters()):,}')

Model: ViT-B-16 pretrained=laion2b_s34b_b88k
Embed dim: 512
Total params: 149,620,737


In [4]:
# ── Dataset classes ─────────────────────────────────────────

class LabeledDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label


class EmbeddingDataset(Dataset):
    """Pre-computed CLIP embeddings — Phase A head training."""
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels     = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


class SimplePathDataset(Dataset):
    """Used only for embedding extraction."""
    def __init__(self, paths, transform):
        self.paths     = paths
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img)


class TestDataset(Dataset):
    def __init__(self, test_dir, transform):
        self.paths = sorted(
            glob.glob(os.path.join(test_dir, '*.jpg')),
            key=lambda x: int(os.path.splitext(os.path.basename(x))[0])
        )
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), os.path.basename(self.paths[idx])


print('Dataset classes defined.')

Dataset classes defined.


In [5]:
# ── Build full sample list ───────────────────────────────────
all_paths, all_labels = [], []
for class_id in range(NUM_CLASSES):
    class_dir = os.path.join(TRAIN_DIR, str(class_id))
    for fname in sorted(os.listdir(class_dir)):
        if fname.endswith('.jpg'):
            all_paths.append(os.path.join(class_dir, fname))
            all_labels.append(class_id)

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
print(f'Total training images: {len(all_paths)} across {NUM_CLASSES} classes')

Total training images: 1079 across 100 classes


In [6]:
# ── Extract cached embeddings (run once) ────────────────────
@torch.no_grad()
def extract_all_embeddings(paths, batch_size=64):
    clip_model.eval()
    ds     = SimplePathDataset(paths, transform=preprocess)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    embs   = []
    for imgs in tqdm(loader, desc='Extracting embeddings'):
        imgs = imgs.to(device)
        embs.append(clip_model.encode_image(imgs).float().cpu())
    return torch.cat(embs, dim=0)

all_embeddings = extract_all_embeddings(all_paths)
print(f'Embeddings shape: {all_embeddings.shape}')  # (N, 512)

Extracting embeddings:   0%|          | 0/17 [00:00<?, ?it/s]

Embeddings shape: torch.Size([1079, 512])


In [7]:
# ── Loss, head, classifier, helper functions ─────────────────

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


class EarlyStopping:
    def __init__(self, patience):
        self.patience   = patience
        self.best_val   = -1.0
        self.counter    = 0
        self.best_epoch = 0
        self.best_state = None
        self._epoch     = 0

    def step(self, val_acc, model):
        self._epoch += 1
        if val_acc > self.best_val:
            self.best_val   = val_acc
            self.counter    = 0
            self.best_epoch = self._epoch
            self.best_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state:
            model.load_state_dict({k: v.to(device) for k, v in self.best_state.items()})


def make_head():
    return nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(EMBED_DIM, NUM_CLASSES)
    ).to(device)


class CLIPClassifier(nn.Module):
    def __init__(self, clip_mdl, freeze_backbone=True):
        super().__init__()
        self.clip = clip_mdl
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(EMBED_DIM, NUM_CLASSES))
        if freeze_backbone:
            for p in self.clip.parameters(): p.requires_grad = False

    def forward(self, x):
        return self.head(self.clip.encode_image(x).float())


def make_full_clf(head_state, n_unfreeze):
    """Build CLIPClassifier, unfreeze the LAST n_unfreeze blocks regardless of backbone depth."""
    clf = CLIPClassifier(clip_model, freeze_backbone=True).to(device)
    clf.head.load_state_dict(head_state)
    if n_unfreeze > 0:
        n_blocks      = len(clf.clip.visual.transformer.resblocks)
        unfreeze_idxs = range(n_blocks - n_unfreeze, n_blocks)  # always the last N
        for block_idx in unfreeze_idxs:
            for p in clf.clip.visual.transformer.resblocks[block_idx].parameters():
                p.requires_grad = True
        for p in clf.clip.visual.ln_post.parameters(): p.requires_grad = True
        clf.clip.visual.proj.requires_grad = True
        print(f'    Unfrozen blocks: {list(unfreeze_idxs)} of {n_blocks}')
    return clf


def train_head_epoch(head, loader, optimizer):
    head.train()
    total_loss, total_correct, n = 0.0, 0, 0
    for embs, labels in loader:
        embs, labels = embs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = head(embs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * embs.size(0)
        total_correct += (out.argmax(1) == labels).sum().item()
        n             += embs.size(0)
    return total_loss / n, total_correct / n


@torch.no_grad()
def eval_head(head, loader):
    head.eval()
    total_loss, total_correct, n = 0.0, 0, 0
    for embs, labels in loader:
        embs, labels = embs.to(device), labels.to(device)
        out  = head(embs)
        loss = criterion(out, labels)
        total_loss    += loss.item() * embs.size(0)
        total_correct += (out.argmax(1) == labels).sum().item()
        n             += embs.size(0)
    return total_loss / n, total_correct / n


def train_one_epoch(clf, loader, optimizer, scheduler=None):
    clf.train()
    total_loss, total_correct, n = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = clf(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * imgs.size(0)
        total_correct += (out.argmax(1) == labels).sum().item()
        n             += imgs.size(0)
    if scheduler: scheduler.step()
    return total_loss / n, total_correct / n


@torch.no_grad()
def evaluate(clf, loader):
    clf.eval()
    total_loss, total_correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out  = clf(imgs)
        loss = criterion(out, labels)
        total_loss    += loss.item() * imgs.size(0)
        total_correct += (out.argmax(1) == labels).sum().item()
        n             += imgs.size(0)
    return total_loss / n, total_correct / n


print(f'Helper functions defined. EMBED_DIM={EMBED_DIM}')

Helper functions defined. EMBED_DIM=512


In [8]:
# ── OPTIONAL: Backbone comparison via head_only CV ───────────
# Run this cell BEFORE the main CV to pick the best backbone.
# Set CLIP_MODEL / CLIP_PRETRAINED in the config cell to the winner,
# then restart from the Load CLIP cell and run the main CV.
#
# Each backbone: load → extract embeddings → 5-fold head_only CV → table.
# Cost: one embedding pass per backbone (seconds-to-minutes), head trains fast.
# ViT-L/14 is large (~900MB) and slow to extract — comment it out if time is short.

import math

BACKBONE_CANDIDATES = [
    {'name': 'ViT-B-32-openai',  'model': 'ViT-B-32', 'pretrained': 'openai'},
    {'name': 'ViT-B-16-openai',  'model': 'ViT-B-16', 'pretrained': 'openai'},
    {'name': 'ViT-B-16-laion2b', 'model': 'ViT-B-16', 'pretrained': 'laion2b_s34b_b88k'},
    # {'name': 'ViT-L-14-openai',  'model': 'ViT-L-14', 'pretrained': 'openai'},  # slow
]

def clear_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

def run_backbone_headonly_cv(bb_cfg):
    bb_model, _, bb_prep = open_clip.create_model_and_transforms(
        bb_cfg['model'], pretrained=bb_cfg['pretrained']
    )
    bb_model = bb_model.to(device)
    bb_model.eval()
    embed_dim = bb_model.visual.output_dim
    print(f'\nBackbone: {bb_cfg["name"]} | embed_dim={embed_dim}')

    @torch.no_grad()
    def extract(paths):
        ds     = SimplePathDataset(paths, transform=bb_prep)
        loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)
        embs   = [bb_model.encode_image(imgs.to(device)).float().cpu() for imgs in tqdm(loader, desc='embed', leave=False)]
        return torch.cat(embs, dim=0)

    embs = extract(all_paths)

    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    fold_vals, fold_epochs = [], []

    for fold_idx, (tr_idx, vl_idx) in enumerate(skf.split(all_paths, all_labels)):
        set_seed(SEED + fold_idx)
        tr_ds = EmbeddingDataset(embs[tr_idx], torch.tensor(all_labels[tr_idx], dtype=torch.long))
        vl_ds = EmbeddingDataset(embs[vl_idx], torch.tensor(all_labels[vl_idx], dtype=torch.long))
        tr_l  = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
        vl_l  = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        head = nn.Sequential(nn.Dropout(0.3), nn.Linear(embed_dim, NUM_CLASSES)).to(device)
        opt  = optim.AdamW(head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
        es   = EarlyStopping(patience=PATIENCE_A)

        for epoch in range(MAX_EPOCHS_A):
            tr_loss, tr_acc = train_head_epoch(head, tr_l, opt)
            vl_loss, vl_acc = eval_head(head, vl_l)
            if es.step(vl_acc, head): break
        es.restore(head)
        fold_vals.append(es.best_val)
        fold_epochs.append(es.best_epoch)
        print(f'  Fold {fold_idx+1}: val {es.best_val:.4f} @ epoch {es.best_epoch}')

    del bb_model
    clear_cache()

    cv_mean = float(np.mean(fold_vals))
    cv_std  = float(np.std(fold_vals))
    se      = cv_std / math.sqrt(K_FOLDS)
    return {'name': bb_cfg['name'], 'embed_dim': embed_dim,
            'cv_mean': cv_mean, 'cv_std': cv_std, 'se': se,
            'median_epoch': int(np.median(fold_epochs))}

backbone_results = []
for bb_cfg in BACKBONE_CANDIDATES:
    backbone_results.append(run_backbone_headonly_cv(bb_cfg))

print('\n── Backbone Comparison (head_only, 5-fold CV) ──────────')
print(f'{"Backbone":<25} {"dim":>5} {"mean":>7} {"std":>7} {"se":>7} {"med_ep":>7}')
print('-' * 60)
for r in backbone_results:
    print(f'{r["name"]:<25} {r["embed_dim"]:>5} {r["cv_mean"]:>7.4f} {r["cv_std"]:>7.4f} {r["se"]:>7.4f} {r["median_epoch"]:>7}')
print('\n→ Set CLIP_MODEL/CLIP_PRETRAINED to the winner, restart from Load CLIP cell.')


Backbone: ViT-B-32-openai | embed_dim=512


embed:   0%|          | 0/17 [00:00<?, ?it/s]

  Fold 1: val 0.7704 @ epoch 17
  Fold 2: val 0.7481 @ epoch 17
  Fold 3: val 0.7407 @ epoch 27
  Fold 4: val 0.7621 @ epoch 33


C:\venvs\ml312\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(



Backbone: ViT-B-16-openai | embed_dim=512


...

open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

C:\venvs\ml312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\boomc\.cache\huggingface\hub\models--laion--CLIP-ViT-B-16-laion2B-s34B-b88K. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



Backbone: ViT-B-16-laion2b | embed_dim=512


embed:   0%|          | 0/17 [00:00<?, ?it/s]

  Fold 1: val 0.8407 @ epoch 13
  Fold 2: val 0.8630 @ epoch 14
  Fold 3: val 0.8444 @ epoch 22
  Fold 4: val 0.8513 @ epoch 27

── Backbone Comparison (head_only, 5-fold CV) ──────────
Backbone                    dim    mean     std      se  med_ep
------------------------------------------------------------
ViT-B-32-openai             512  0.7553  0.0116  0.0058      22
ViT-B-16-openai             512  0.7933  0.0017  0.0009      23
ViT-B-16-laion2b            512  0.8499  0.0085  0.0042      18

→ Set CLIP_MODEL/CLIP_PRETRAINED to the winner, restart from Load CLIP cell.


In [8]:
# ── CV runner ────────────────────────────────────────────────
def run_cv_config(config):
    config_name = config['name']
    n_unfreeze  = config['n_unfreeze']
    lr_backbone = config['lr_backbone']
    fold_best_vals, fold_best_epochs, fold_a_epochs = [], [], []

    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

    for fold_idx, (tr_idx, vl_idx) in enumerate(skf.split(all_paths, all_labels)):
        print(f'\n--- Config: {config_name} | Fold {fold_idx+1}/{K_FOLDS} ---')
        set_seed(SEED + fold_idx)

        # ── Phase A: head on cached embeddings ──
        tr_labs = torch.tensor(all_labels[tr_idx], dtype=torch.long)
        vl_labs = torch.tensor(all_labels[vl_idx], dtype=torch.long)
        tr_emb_ds = EmbeddingDataset(all_embeddings[tr_idx], tr_labs)
        vl_emb_ds = EmbeddingDataset(all_embeddings[vl_idx], vl_labs)
        tr_emb_loader = DataLoader(tr_emb_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
        vl_emb_loader = DataLoader(vl_emb_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        head  = make_head()
        opt_a = optim.AdamW(head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
        es_a  = EarlyStopping(patience=PATIENCE_A)

        print(f'  Phase A (cached, max {MAX_EPOCHS_A} epochs, patience {PATIENCE_A}):')
        for epoch in range(MAX_EPOCHS_A):
            tr_loss, tr_acc = train_head_epoch(head, tr_emb_loader, opt_a)
            vl_loss, vl_acc = eval_head(head, vl_emb_loader)
            stop = es_a.step(vl_acc, head)
            print(f'    [{epoch+1:02d}] train {tr_acc:.4f} | val {vl_acc:.4f} | gap {tr_acc-vl_acc:.4f}')
            if stop:
                print(f'    Early stop. Best val {es_a.best_val:.4f} @ epoch {es_a.best_epoch}')
                break
        es_a.restore(head)
        fold_a_epochs.append(es_a.best_epoch)

        if n_unfreeze == 0:
            fold_best_vals.append(es_a.best_val)
            fold_best_epochs.append(es_a.best_epoch)
            continue

        # ── Phase B: fine-tune on raw images ──
        clip_model.load_state_dict({k: v.to(device) for k, v in original_clip_state.items()})

        train_tf = transforms.Compose([transforms.RandomHorizontalFlip(), preprocess]) if USE_AUG else preprocess
        tr_samples = list(zip(all_paths[tr_idx].tolist(), all_labels[tr_idx].tolist()))
        vl_samples = list(zip(all_paths[vl_idx].tolist(), all_labels[vl_idx].tolist()))
        tr_ds = LabeledDataset(tr_samples, transform=train_tf)
        vl_ds = LabeledDataset(vl_samples, transform=preprocess)
        tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
        vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        clf   = make_full_clf(head.state_dict(), n_unfreeze)
        opt_b = optim.AdamW([
            {'params': [p for p in clf.clip.parameters() if p.requires_grad], 'lr': lr_backbone},
            {'params': clf.head.parameters(), 'lr': LR_HEAD},
        ], weight_decay=WEIGHT_DECAY)
        sch_b = optim.lr_scheduler.CosineAnnealingLR(opt_b, T_max=MAX_EPOCHS_B)
        es_b  = EarlyStopping(patience=PATIENCE_B)

        print(f'  Phase B (last {n_unfreeze} blocks @ lr={lr_backbone}, max {MAX_EPOCHS_B} epochs, patience {PATIENCE_B}):')
        for epoch in range(MAX_EPOCHS_B):
            tr_loss, tr_acc = train_one_epoch(clf, tr_loader, opt_b, sch_b)
            vl_loss, vl_acc = evaluate(clf, vl_loader)
            stop = es_b.step(vl_acc, clf)
            print(f'    [{epoch+1:02d}] train {tr_acc:.4f} | val {vl_acc:.4f} | gap {tr_acc-vl_acc:.4f}')
            if stop:
                print(f'    Early stop. Best val {es_b.best_val:.4f} @ epoch {es_b.best_epoch}')
                break
        es_b.restore(clf)

        torch.save({'model_state_dict': clf.state_dict(), 'val_acc': es_b.best_val},
                   os.path.join(CKPT_DIR, f'{config_name}_fold{fold_idx}_best.pt'))

        fold_best_vals.append(es_b.best_val)
        fold_best_epochs.append(es_b.best_epoch)

        clip_model.load_state_dict({k: v.to(device) for k, v in original_clip_state.items()})

    return {
        'name':           config_name,
        'n_unfreeze':     n_unfreeze,
        'lr_backbone':    lr_backbone,
        'fold_vals':      fold_best_vals,
        'fold_epochs':    fold_best_epochs,
        'fold_a_epochs':  fold_a_epochs,
        'cv_mean':        float(np.mean(fold_best_vals)),
        'cv_std':         float(np.std(fold_best_vals)),
        'median_epoch':   int(np.median(fold_best_epochs)),
        'median_a_epoch': int(np.median(fold_a_epochs)),
    }


print('CV runner defined.')

CV runner defined.


In [16]:
# ── Run CV for all configs ───────────────────────────────────
all_results = []
for config in CONFIGS:
    print(f'\n{"="*60}')
    print(f'Running CV: {config["name"]}')
    print(f'{"="*60}')
    result = run_cv_config(config)
    all_results.append(result)
    print(f'\n{config["name"]} → mean {result["cv_mean"]:.4f} ± {result["cv_std"]:.4f} | median epoch {result["median_epoch"]}')


Running CV: head_only

--- Config: head_only | Fold 1/4 ---
  Phase A (cached, max 60 epochs, patience 10):
    [01] train 0.0865 | val 0.2704 | gap -0.1838
    [02] train 0.3498 | val 0.4481 | gap -0.0983
    [03] train 0.5711 | val 0.5519 | gap 0.0192
    [04] train 0.7058 | val 0.6556 | gap 0.0503
    [05] train 0.7911 | val 0.6926 | gap 0.0985
    [06] train 0.8529 | val 0.7222 | gap 0.1307
    [07] train 0.9085 | val 0.7704 | gap 0.1382
    [08] train 0.9234 | val 0.7852 | gap 0.1382
    [09] train 0.9555 | val 0.8185 | gap 0.1370
    [10] train 0.9654 | val 0.8222 | gap 0.1432
    [11] train 0.9691 | val 0.8222 | gap 0.1469
    [12] train 0.9753 | val 0.8370 | gap 0.1382
    [13] train 0.9815 | val 0.8407 | gap 0.1407
    [14] train 0.9827 | val 0.8296 | gap 0.1531
    [15] train 0.9864 | val 0.8370 | gap 0.1494
    [16] train 0.9901 | val 0.8407 | gap 0.1494
    [17] train 0.9889 | val 0.8370 | gap 0.1518
    [18] train 0.9913 | val 0.8370 | gap 0.1543
    [19] train 0.9913 | v

  0%|          | 0/13 [00:00<?, ?it/s]

    [01] train 0.9666 | val 0.8296 | gap 0.1370


  0%|          | 0/13 [00:00<?, ?it/s]

    [02] train 0.9827 | val 0.8370 | gap 0.1457


...

    [05] train 1.0000 | val 0.8364 | gap 0.1636


  0%|          | 0/13 [00:00<?, ?it/s]

    [06] train 1.0000 | val 0.8476 | gap 0.1524


  0%|          | 0/13 [00:00<?, ?it/s]

    [07] train 1.0000 | val 0.8476 | gap 0.1524
    Early stop. Best val 0.8587 @ epoch 2

unfreeze_4_1e5 → mean 0.8647 ± 0.0143 | median epoch 2


In [17]:
# ── Results table + 1-std-error model selection ──────────────
import math

print('\n── CV Results ──────────────────────────────────────────────────────')
print(f'{"Config":<20} {"unfreeze":>9} {"lr_bb":>7} {"mean":>7} {"std":>7} {"se":>7} {"med_B":>7} {"med_A":>7} {"folds"}')
print('-' * 95)
for r in all_results:
    se        = r['cv_std'] / math.sqrt(K_FOLDS)
    folds_str = ' '.join(f'{v:.4f}' for v in r['fold_vals'])
    print(f'{r["name"]:<20} {r["n_unfreeze"]:>9} {r["lr_backbone"]:>7.0e} {r["cv_mean"]:>7.4f} {r["cv_std"]:>7.4f} {se:>7.4f} {r["median_epoch"]:>7} {r["median_a_epoch"]:>7} {folds_str}')

# 1-std-error rule (strict: std / sqrt(K))
best_mean = max(r['cv_mean'] for r in all_results)
best_se   = next(r['cv_std'] / math.sqrt(K_FOLDS) for r in all_results if r['cv_mean'] == best_mean)
threshold = best_mean - best_se

candidates = [r for r in all_results if r['cv_mean'] >= threshold]
selected   = min(candidates, key=lambda r: r['n_unfreeze'])

print(f'\n── Model Selection (1-SE rule, strict: std/√{K_FOLDS}) ────────────')
print(f'Best mean:  {best_mean:.4f}')
print(f'SE:         {best_se:.4f}')
print(f'Threshold:  {threshold:.4f}')
print(f'Candidates: {[r["name"] for r in candidates]}')
print(f'Selected:   {selected["name"]} (simplest within 1 SE of best)')
print(f'Phase A budget: {selected["median_a_epoch"]} epochs')
print(f'Phase B budget: {selected["median_epoch"]} epochs')


── CV Results ──────────────────────────────────────────────────────
Config                unfreeze   lr_bb    mean     std      se   med_B   med_A folds
-----------------------------------------------------------------------------------------------
head_only                    0   1e-06  0.8499  0.0085  0.0042      18      18 0.8407 0.8630 0.8444 0.8513
unfreeze_2_1e6               2   1e-06  0.8545  0.0144  0.0072       3      18 0.8407 0.8778 0.8444 0.8550
unfreeze_2_1e5               2   1e-05  0.8619  0.0145  0.0073       2      18 0.8630 0.8852 0.8481 0.8513
unfreeze_4_1e5               4   1e-05  0.8647  0.0143  0.0071       2      18 0.8593 0.8889 0.8519 0.8587

── Model Selection (1-SE rule, strict: std/√4) ────────────
Best mean:  0.8647
SE:         0.0071
Threshold:  0.8575
Candidates: ['unfreeze_2_1e5', 'unfreeze_4_1e5']
Selected:   unfreeze_2_1e5 (simplest within 1 SE of best)
Phase A budget: 18 epochs
Phase B budget: 2 epochs


In [18]:
# ── Final retrain on ALL data ────────────────────────────────
# Phase A budget = median_a_epoch from CV
# Phase B budget = median_epoch from CV
# T_max=MAX_EPOCHS_B matches CV's LR schedule

FINAL_CONFIG   = selected
FINAL_EPOCHS_A = FINAL_CONFIG['median_a_epoch']
FINAL_EPOCHS_B = FINAL_CONFIG['median_epoch'] if FINAL_CONFIG['n_unfreeze'] > 0 else 0
FINAL_LR_BB    = FINAL_CONFIG['lr_backbone']

print(f'Final retrain: {FINAL_CONFIG["name"]}')
print(f'Phase A: {FINAL_EPOCHS_A} epochs (CV median)')
print(f'Phase B: {FINAL_EPOCHS_B} epochs (CV median) @ lr_backbone={FINAL_LR_BB}')

clip_model.load_state_dict({k: v.to(device) for k, v in original_clip_state.items()})
set_seed(SEED)

all_labs_tensor = torch.tensor(all_labels, dtype=torch.long)
all_emb_ds      = EmbeddingDataset(all_embeddings, all_labs_tensor)
all_emb_loader  = DataLoader(all_emb_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

final_head = make_head()
opt_fa     = optim.AdamW(final_head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

print(f'\nPhase A — all data, {FINAL_EPOCHS_A} epochs:')
for epoch in range(FINAL_EPOCHS_A):
    tr_loss, tr_acc = train_head_epoch(final_head, all_emb_loader, opt_fa)
    print(f'  [{epoch+1:02d}] train {tr_acc:.4f}')

if FINAL_CONFIG['n_unfreeze'] > 0:
    train_tf = transforms.Compose([transforms.RandomHorizontalFlip(), preprocess]) if USE_AUG else preprocess
    all_samples    = list(zip(all_paths.tolist(), all_labels.tolist()))
    all_raw_ds     = LabeledDataset(all_samples, transform=train_tf)
    all_raw_loader = DataLoader(all_raw_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

    final_clf = make_full_clf(final_head.state_dict(), FINAL_CONFIG['n_unfreeze'])
    opt_fb    = optim.AdamW([
        {'params': [p for p in final_clf.clip.parameters() if p.requires_grad], 'lr': FINAL_LR_BB},
        {'params': final_clf.head.parameters(), 'lr': LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)
    sch_fb = optim.lr_scheduler.CosineAnnealingLR(opt_fb, T_max=MAX_EPOCHS_B)

    print(f'\nPhase B — all data, {FINAL_EPOCHS_B} epochs (last {FINAL_CONFIG["n_unfreeze"]} blocks):')
    for epoch in range(FINAL_EPOCHS_B):
        tr_loss, tr_acc = train_one_epoch(final_clf, all_raw_loader, opt_fb, sch_fb)
        print(f'  [{epoch+1:02d}] train {tr_acc:.4f}')
else:
    final_clf = CLIPClassifier(clip_model, freeze_backbone=True).to(device)
    final_clf.head.load_state_dict(final_head.state_dict())

torch.save({'model_state_dict': final_clf.state_dict(), 'config': FINAL_CONFIG['name']},
           os.path.join(CKPT_DIR, 'final_model.pt'))
print('\nSaved: checkpoints_kfold/final_model.pt')

Final retrain: unfreeze_2_1e5
Phase A: 18 epochs (CV median)
Phase B: 2 epochs (CV median) @ lr_backbone=1e-05

Phase A — all data, 18 epochs:
  [01] train 0.1205
  [02] train 0.4421
  [03] train 0.6284
  [04] train 0.7665
  [05] train 0.8508
  [06] train 0.8804
  [07] train 0.9249
  [08] train 0.9500
  [09] train 0.9481
  [10] train 0.9555
  [11] train 0.9648
  [12] train 0.9685
  [13] train 0.9741
  [14] train 0.9805
  [15] train 0.9778
  [16] train 0.9824
  [17] train 0.9889
  [18] train 0.9870
    Unfrozen blocks: [10, 11] of 12

Phase B — all data, 2 epochs (last 2 blocks):


  0%|          | 0/17 [00:00<?, ?it/s]

  [01] train 0.9815


  0%|          | 0/17 [00:00<?, ?it/s]

  [02] train 0.9935

Saved: checkpoints_kfold/final_model.pt


In [10]:
# ── Submission ───────────────────────────────────────────────
test_ds     = TestDataset(TEST_DIR, transform=preprocess)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

ckpt = torch.load(os.path.join(CKPT_DIR, 'final_model.pt'), map_location=device)
final_clf = CLIPClassifier(clip_model, freeze_backbone=True).to(device)
final_clf.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded final model: {ckpt["config"]}')

final_clf.eval()
ids, preds = [], []
with torch.no_grad():
    for imgs, names in tqdm(test_loader):
        imgs = imgs.to(device)
        out  = final_clf(imgs)
        ids.extend(names)
        preds.extend(out.argmax(1).cpu().tolist())

sub = pd.DataFrame({'ID': ids, 'Label': preds})
sub.to_csv('submission_kfold.csv', index=False)
print(sub.head(10))
print(f'Saved submission_kfold.csv ({len(sub)} rows)')

Loaded final model: unfreeze_2_1e5


  0%|          | 0/17 [00:00<?, ?it/s]

      ID  Label
0  0.jpg     62
1  1.jpg     43
2  2.jpg     38
3  3.jpg     51
4  4.jpg     42
5  5.jpg     89
6  6.jpg      3
7  7.jpg     28
8  8.jpg     71
9  9.jpg     71
Saved submission_kfold.csv (1036 rows)


In [11]:
# ── Fold-ensemble inference ─────────────────────────────────────────────
ENSEMBLE_CONFIG     = 'unfreeze_2_1e5'
ENSEMBLE_N_UNFREEZE = 2

ckpt_paths = sorted(glob.glob(os.path.join(CKPT_DIR, f'{ENSEMBLE_CONFIG}_fold*_best.pt')))
assert len(ckpt_paths) == 4, f'Expected 4 checkpoints, found {len(ckpt_paths)}: {ckpt_paths}'
print(f'Found {len(ckpt_paths)} checkpoints:')
for p in ckpt_paths:
    print(f'  {os.path.basename(p)}')

all_fold_logits = []
all_fold_preds  = []

for ckpt_path in ckpt_paths:
    # Reset backbone before each fold — required because clf.clip IS clip_model;
    # loading the checkpoint's state_dict mutates the global in-place.
    clip_model.load_state_dict({k: v.to(device) for k, v in original_clip_state.items()})

    ckpt = torch.load(ckpt_path, map_location=device)
    head_state = {k[len('head.'):]: v for k, v in ckpt['model_state_dict'].items()
                  if k.startswith('head.')}
    clf = make_full_clf(head_state, ENSEMBLE_N_UNFREEZE)
    clf.load_state_dict({k: v.to(device) for k, v in ckpt['model_state_dict'].items()})
    clf.eval()

    fold_logits = []
    with torch.no_grad():
        for imgs, _ in tqdm(test_loader, desc=f'  {os.path.basename(ckpt_path)}', leave=False):
            imgs = imgs.to(device)
            fold_logits.append(clf(imgs).cpu())

    fold_tensor = torch.cat(fold_logits, dim=0)          # (N, 100)
    all_fold_logits.append(fold_tensor)
    all_fold_preds.append(fold_tensor.argmax(dim=1))

    del clf, fold_logits, fold_tensor
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

# Sum logits → argmax (equivalent to mean for argmax; avoids a divide)
ensemble_preds = torch.stack(all_fold_logits, dim=0).sum(dim=0).argmax(dim=1).tolist()

# ID format matches Submission cell exactly (basename of sorted test paths)
ensemble_ids = [os.path.basename(p) for p in test_ds.paths]
sub_ensemble = pd.DataFrame({'ID': ensemble_ids, 'Label': ensemble_preds})
sub_ensemble.to_csv('submission_ensemble.csv', index=False)

print(f'\nEnsembled {len(ckpt_paths)} models | {len(sub_ensemble)} rows')
print(sub_ensemble.head(10))

print('\nAgreement rates (ensemble vs each fold):')
ens_arr = np.array(ensemble_preds)
for i, fp in enumerate(all_fold_preds):
    print(f'  fold{i}: {(ens_arr == fp.numpy()).mean():.4f}')

print('\nSaved: submission_ensemble.csv')

Found 4 checkpoints:
  unfreeze_2_1e5_fold0_best.pt
  unfreeze_2_1e5_fold1_best.pt
  unfreeze_2_1e5_fold2_best.pt
  unfreeze_2_1e5_fold3_best.pt
    Unfrozen blocks: [10, 11] of 12


  unfreeze_2_1e5_fold0_best.pt:   0%|          | 0/17 [00:00<?, ?it/s]

    Unfrozen blocks: [10, 11] of 12


  unfreeze_2_1e5_fold1_best.pt:   0%|          | 0/17 [00:00<?, ?it/s]

    Unfrozen blocks: [10, 11] of 12


  unfreeze_2_1e5_fold2_best.pt:   0%|          | 0/17 [00:00<?, ?it/s]

    Unfrozen blocks: [10, 11] of 12


  unfreeze_2_1e5_fold3_best.pt:   0%|          | 0/17 [00:00<?, ?it/s]


Ensembled 4 models | 1036 rows
      ID  Label
0  0.jpg     62
1  1.jpg     43
2  2.jpg     38
3  3.jpg     51
4  4.jpg     42
5  5.jpg     89
6  6.jpg      3
7  7.jpg     28
8  8.jpg     71
9  9.jpg     71

Agreement rates (ensemble vs each fold):
  fold0: 0.9440
  fold1: 0.9228
  fold2: 0.9276
  fold3: 0.9180

Saved: submission_ensemble.csv


In [9]:
# ── Stage 1: ViT-L/14 — load, embed, head-only CV ────────────────────────
import math

L14_MODEL      = 'ViT-L-14'
L14_PRETRAINED = 'laion2b_s32b_b82k'
L14_EMBED_PATH = 'l14_embeddings.pt'
L14_BATCH_SIZE = 32  # L/14 is ~1.6 GB; smaller batch for memory

# Move B/16 off GPU before loading L/14.
# NOTE: clip_model now lives on CPU. If you re-run any B/16 cell below
# (e.g. the ensemble cell), add clip_model.to(device) before it or you
# will get a device-mismatch error on the first forward pass.
clip_model.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()

l14_model, _, l14_preprocess = open_clip.create_model_and_transforms(
    L14_MODEL, pretrained=L14_PRETRAINED
)
l14_model = l14_model.to(device)
l14_model.eval()
l14_embed_dim = l14_model.visual.output_dim  # 768 — never hardcode

# Snapshot original L/14 weights — restored before each Phase B fold (Stage 2)
l14_original_state = {k: v.clone().cpu() for k, v in l14_model.state_dict().items()}

print(f'Model: {L14_MODEL} pretrained={L14_PRETRAINED}')
print(f'L/14 embed dim: {l14_embed_dim}')
print(f'L/14 total params: {sum(p.numel() for p in l14_model.parameters()):,}')


class L14Classifier(nn.Module):
    """CLIPClassifier equivalent for L/14 — takes embed_dim explicitly, not EMBED_DIM global."""
    def __init__(self, clip_mdl, embed_dim, freeze_backbone=True):
        super().__init__()
        self.clip = clip_mdl
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(embed_dim, NUM_CLASSES))
        if freeze_backbone:
            for p in self.clip.parameters(): p.requires_grad = False
    def forward(self, x):
        return self.head(self.clip.encode_image(x).float())


def make_l14_clf(head_state, n_unfreeze):
    """make_full_clf equivalent for L/14 — uses l14_model and l14_embed_dim."""
    clf = L14Classifier(l14_model, l14_embed_dim, freeze_backbone=True).to(device)
    clf.head.load_state_dict(head_state)
    if n_unfreeze > 0:
        n_blocks      = len(clf.clip.visual.transformer.resblocks)
        unfreeze_idxs = range(n_blocks - n_unfreeze, n_blocks)
        for block_idx in unfreeze_idxs:
            for p in clf.clip.visual.transformer.resblocks[block_idx].parameters():
                p.requires_grad = True
        for p in clf.clip.visual.ln_post.parameters(): p.requires_grad = True
        clf.clip.visual.proj.requires_grad = True
        print(f'    Unfrozen blocks: {list(unfreeze_idxs)} of {n_blocks}')
    return clf


# ── Extract or load L/14 embeddings ──────────────────────────────────────
if os.path.exists(L14_EMBED_PATH):
    l14_embeddings = torch.load(L14_EMBED_PATH, map_location='cpu')
    print(f'\nLoaded L/14 embeddings from disk: {l14_embeddings.shape}')
else:
    print('\nExtracting L/14 embeddings (one-time, slow)...')
    l14_model.eval()
    _ds     = SimplePathDataset(all_paths, transform=l14_preprocess)
    _loader = DataLoader(_ds, batch_size=L14_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    _embs   = []
    with torch.no_grad():
        for _imgs in tqdm(_loader, desc='L/14 embeddings'):
            _embs.append(l14_model.encode_image(_imgs.to(device)).float().cpu())
    l14_embeddings = torch.cat(_embs, dim=0)
    torch.save(l14_embeddings, L14_EMBED_PATH)
    del _ds, _loader, _embs
    print(f'Saved L/14 embeddings: {l14_embeddings.shape} → {L14_EMBED_PATH}')


# ── head-only CV — identical folds to B/16 ───────────────────────────────
print(f'\n── L/14 head_only CV ({K_FOLDS}-fold) ──────────────────────────────')
_skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
l14_head_fold_vals, l14_head_fold_epochs = [], []

for _fold_idx, (_tr_idx, _vl_idx) in enumerate(_skf.split(all_paths, all_labels)):
    set_seed(SEED + _fold_idx)
    _tr_ds = EmbeddingDataset(l14_embeddings[_tr_idx],
                               torch.tensor(all_labels[_tr_idx], dtype=torch.long))
    _vl_ds = EmbeddingDataset(l14_embeddings[_vl_idx],
                               torch.tensor(all_labels[_vl_idx], dtype=torch.long))
    _tr_loader = DataLoader(_tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    _vl_loader = DataLoader(_vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    _head = nn.Sequential(nn.Dropout(0.3), nn.Linear(l14_embed_dim, NUM_CLASSES)).to(device)
    _opt  = optim.AdamW(_head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
    _es   = EarlyStopping(patience=PATIENCE_A)

    print(f'\n--- Fold {_fold_idx+1}/{K_FOLDS} ---')
    for _epoch in range(MAX_EPOCHS_A):
        _tr_loss, _tr_acc = train_head_epoch(_head, _tr_loader, _opt)
        _vl_loss, _vl_acc = eval_head(_head, _vl_loader)
        _stop = _es.step(_vl_acc, _head)
        print(f'  [{_epoch+1:02d}] train {_tr_acc:.4f} | val {_vl_acc:.4f} | gap {_tr_acc-_vl_acc:.4f}')
        if _stop:
            print(f'  Early stop. Best val {_es.best_val:.4f} @ epoch {_es.best_epoch}')
            break
    l14_head_fold_vals.append(_es.best_val)
    l14_head_fold_epochs.append(_es.best_epoch)

l14_head_mean   = float(np.mean(l14_head_fold_vals))
l14_head_std    = float(np.std(l14_head_fold_vals))
l14_head_se     = l14_head_std / math.sqrt(K_FOLDS)
l14_head_med_ep = int(np.median(l14_head_fold_epochs))

_b16_mean, _b16_std, _b16_se = 0.8499, 0.0085, 0.0042
print(f'\n── Stage 1 Summary ─────────────────────────────────────────────────')
print(f'{"Backbone":<28} {"mean":>7} {"std":>7} {"se":>7} {"med_ep":>7}')
print('-' * 58)
print(f'{"B/16-laion2b (head_only)":<28} {_b16_mean:>7.4f} {_b16_std:>7.4f} {_b16_se:>7.4f} {18:>7}')
print(f'{"L/14-laion2b (head_only)":<28} {l14_head_mean:>7.4f} {l14_head_std:>7.4f} {l14_head_se:>7.4f} {l14_head_med_ep:>7}')
_stage2_thresh = _b16_mean + 2 * _b16_se
print(f'\nDecision threshold for Stage 2: B/16 mean + 2\u00d7SE = {_b16_mean:.4f} + 2\u00d7{_b16_se:.4f} = {_stage2_thresh:.4f}')
print(f'L/14 head_only {"PASSES \u2713" if l14_head_mean > _stage2_thresh else "does not pass \u2717"} '
      f'({"proceed" if l14_head_mean > _stage2_thresh else "stop"} \u2192 flip RUN_L14_UNFREEZE)')


open_clip_pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

C:\venvs\ml312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\boomc\.cache\huggingface\hub\models--laion--CLIP-ViT-L-14-laion2B-s32B-b82K. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Model: ViT-L-14 pretrained=laion2b_s32b_b82k
L/14 embed dim: 768
L/14 total params: 427,616,513

Extracting L/14 embeddings (one-time, slow)...


L/14 embeddings:   0%|          | 0/34 [00:00<?, ?it/s]

Saved L/14 embeddings: torch.Size([1079, 768]) → l14_embeddings.pt

── L/14 head_only CV (4-fold) ──────────────────────────────

--- Fold 1/4 ---
  [01] train 0.1273 | val 0.4370 | gap -0.3097
  [02] train 0.5439 | val 0.6593 | gap -0.1154
  [03] train 0.7960 | val 0.7704 | gap 0.0257
  [04] train 0.8789 | val 0.8296 | gap 0.0492
  [05] train 0.9295 | val 0.8333 | gap 0.0962
  [06] train 0.9654 | val 0.8556 | gap 0.1098
  [07] train 0.9740 | val 0.8593 | gap 0.1148
  [08] train 0.9852 | val 0.8481 | gap 0.1370
  [09] train 0.9852 | val 0.8704 | gap 0.1148
  [10] train 0.9913 | val 0.8556 | gap 0.1358
  [11] train 0.9926 | val 0.8667 | gap 0.1259
  [12] train 0.9913 | val 0.8667 | gap 0.1247
  [13] train 0.9975 | val 0.8630 | gap 0.1346
  [14] train 0.9926 | val 0.8593 | gap 0.1333
  [15] train 0.9975 | val 0.8667 | gap 0.1309
  [16] train 0.9975 | val 0.8667 | gap 0.1309
  [17] train 0.9975 | val 0.8667 | gap 0.1309
  [18] train 0.9975 | val 0.8704 | gap 0.1272
  [19] train 0.9988 | v

In [10]:
# ── Stage 2: ViT-L/14 unfreeze_2 @ lr=1e-5 CV (GATED) ───────────────────
# Flip RUN_L14_UNFREEZE = True only if Stage 1 L/14 mean exceeds B/16
# head_only mean by ~2 SE: l14_head_mean > 0.8499 + 2×0.0042 ≈ 0.8583
RUN_L14_UNFREEZE = True

if not RUN_L14_UNFREEZE:
    print('GATED — set RUN_L14_UNFREEZE = True to run Stage 2.')
    print(f'Proceed only if Stage 1 L/14 mean > 0.8499 + 2×0.0042 = {0.8499+2*0.0042:.4f}')
else:
    L14_UF_N         = 2
    L14_UF_LR        = 1e-5
    L14_UF_CKPT_NAME = 'l14_unfreeze_2_1e5'

    _skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    l14_uf_fold_vals, l14_uf_fold_epochs, l14_uf_fold_a_epochs = [], [], []

    for _fold_idx, (_tr_idx, _vl_idx) in enumerate(_skf.split(all_paths, all_labels)):
        print(f'\n--- L/14 unfreeze_2 | Fold {_fold_idx+1}/{K_FOLDS} ---')
        set_seed(SEED + _fold_idx)

        # Phase A: head on L/14 cached embeddings
        _tr_emb_ds = EmbeddingDataset(l14_embeddings[_tr_idx],
                                       torch.tensor(all_labels[_tr_idx], dtype=torch.long))
        _vl_emb_ds = EmbeddingDataset(l14_embeddings[_vl_idx],
                                       torch.tensor(all_labels[_vl_idx], dtype=torch.long))
        _tr_emb_loader = DataLoader(_tr_emb_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
        _vl_emb_loader = DataLoader(_vl_emb_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        _l14_head = nn.Sequential(nn.Dropout(0.3), nn.Linear(l14_embed_dim, NUM_CLASSES)).to(device)
        _opt_a    = optim.AdamW(_l14_head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
        _es_a     = EarlyStopping(patience=PATIENCE_A)

        print(f'  Phase A (cached, max {MAX_EPOCHS_A} epochs, patience {PATIENCE_A}):')
        for _epoch in range(MAX_EPOCHS_A):
            _tr_loss, _tr_acc = train_head_epoch(_l14_head, _tr_emb_loader, _opt_a)
            _vl_loss, _vl_acc = eval_head(_l14_head, _vl_emb_loader)
            _stop = _es_a.step(_vl_acc, _l14_head)
            print(f'    [{_epoch+1:02d}] train {_tr_acc:.4f} | val {_vl_acc:.4f} | gap {_tr_acc-_vl_acc:.4f}')
            if _stop:
                print(f'    Early stop. Best val {_es_a.best_val:.4f} @ epoch {_es_a.best_epoch}')
                break
        _es_a.restore(_l14_head)
        l14_uf_fold_a_epochs.append(_es_a.best_epoch)

        # Phase B: fine-tune last 2 blocks of L/14 on raw images
        # USE_AUG=False in this project, so plain l14_preprocess matches B/16 Phase B exactly
        l14_model.load_state_dict({k: v.to(device) for k, v in l14_original_state.items()})

        _tr_samples = list(zip(all_paths[_tr_idx].tolist(), all_labels[_tr_idx].tolist()))
        _vl_samples = list(zip(all_paths[_vl_idx].tolist(), all_labels[_vl_idx].tolist()))
        _tr_ds  = LabeledDataset(_tr_samples, transform=l14_preprocess)
        _vl_ds  = LabeledDataset(_vl_samples, transform=l14_preprocess)
        _tr_loader = DataLoader(_tr_ds, batch_size=L14_BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
        _vl_loader = DataLoader(_vl_ds, batch_size=L14_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        _clf   = make_l14_clf(_l14_head.state_dict(), L14_UF_N)
        _opt_b = optim.AdamW([
            {'params': [p for p in _clf.clip.parameters() if p.requires_grad], 'lr': L14_UF_LR},
            {'params': _clf.head.parameters(), 'lr': LR_HEAD},
        ], weight_decay=WEIGHT_DECAY)
        _sch_b = optim.lr_scheduler.CosineAnnealingLR(_opt_b, T_max=MAX_EPOCHS_B)
        _es_b  = EarlyStopping(patience=PATIENCE_B)

        print(f'  Phase B (last {L14_UF_N} blocks @ lr={L14_UF_LR}, max {MAX_EPOCHS_B} epochs, patience {PATIENCE_B}):')
        for _epoch in range(MAX_EPOCHS_B):
            _tr_loss, _tr_acc = train_one_epoch(_clf, _tr_loader, _opt_b, _sch_b)
            _vl_loss, _vl_acc = evaluate(_clf, _vl_loader)
            _stop = _es_b.step(_vl_acc, _clf)
            print(f'    [{_epoch+1:02d}] train {_tr_acc:.4f} | val {_vl_acc:.4f} | gap {_tr_acc-_vl_acc:.4f}')
            if _stop:
                print(f'    Early stop. Best val {_es_b.best_val:.4f} @ epoch {_es_b.best_epoch}')
                break
        _es_b.restore(_clf)

        torch.save({'model_state_dict': _clf.state_dict(), 'val_acc': _es_b.best_val},
                   os.path.join(CKPT_DIR, f'{L14_UF_CKPT_NAME}_fold{_fold_idx}_best.pt'))

        l14_uf_fold_vals.append(_es_b.best_val)
        l14_uf_fold_epochs.append(_es_b.best_epoch)

        l14_model.load_state_dict({k: v.to(device) for k, v in l14_original_state.items()})
        del _clf
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        elif torch.backends.mps.is_available(): torch.mps.empty_cache()

    l14_uf_mean   = float(np.mean(l14_uf_fold_vals))
    l14_uf_std    = float(np.std(l14_uf_fold_vals))
    l14_uf_se     = l14_uf_std / math.sqrt(K_FOLDS)
    l14_uf_med_ep = int(np.median(l14_uf_fold_epochs))

    _b16_uf_mean, _b16_uf_std, _b16_uf_se = 0.8619, 0.0145, 0.0073
    print(f'\n── Stage 2 Summary ──────────────────────────────────────────────────────')
    print(f'{"Config":<28} {"mean":>7} {"std":>7} {"se":>7} {"med_B":>7}')
    print('-' * 58)
    print(f'{"B/16 unfreeze_2_1e5":<28} {_b16_uf_mean:>7.4f} {_b16_uf_std:>7.4f} {_b16_uf_se:>7.4f} {2:>7}')
    print(f'{"L/14 unfreeze_2_1e5":<28} {l14_uf_mean:>7.4f} {l14_uf_std:>7.4f} {l14_uf_se:>7.4f} {l14_uf_med_ep:>7}')
    _stage3_thresh = _b16_uf_mean + 2 * _b16_uf_se
    print(f'\nDecision threshold for Stage 3: B/16 mean + 2×SE = {_b16_uf_mean:.4f} + 2×{_b16_uf_se:.4f} = {_stage3_thresh:.4f}')
    print(f'L/14 unfreeze {"PASSES ✓" if l14_uf_mean > _stage3_thresh else "does not pass ✗"} '
          f'({"proceed" if l14_uf_mean > _stage3_thresh else "stop"} → flip RUN_L14_SUBMISSION)')



--- L/14 unfreeze_2 | Fold 1/4 ---
  Phase A (cached, max 60 epochs, patience 10):
    [01] train 0.1273 | val 0.4370 | gap -0.3097
    [02] train 0.5439 | val 0.6593 | gap -0.1154
    [03] train 0.7960 | val 0.7704 | gap 0.0257
    [04] train 0.8789 | val 0.8296 | gap 0.0492
    [05] train 0.9295 | val 0.8333 | gap 0.0962
    [06] train 0.9654 | val 0.8556 | gap 0.1098
    [07] train 0.9740 | val 0.8593 | gap 0.1148
    [08] train 0.9852 | val 0.8481 | gap 0.1370
    [09] train 0.9852 | val 0.8704 | gap 0.1148
    [10] train 0.9913 | val 0.8556 | gap 0.1358
    [11] train 0.9926 | val 0.8667 | gap 0.1259
    [12] train 0.9913 | val 0.8667 | gap 0.1247
    [13] train 0.9975 | val 0.8630 | gap 0.1346
    [14] train 0.9926 | val 0.8593 | gap 0.1333
    [15] train 0.9975 | val 0.8667 | gap 0.1309
    [16] train 0.9975 | val 0.8667 | gap 0.1309
    [17] train 0.9975 | val 0.8667 | gap 0.1309
    [18] train 0.9975 | val 0.8704 | gap 0.1272
    [19] train 0.9988 | val 0.8630 | gap 0.1358
  

  0%|          | 0/26 [00:00<?, ?it/s]

    [01] train 0.9728 | val 0.8630 | gap 0.1098


  0%|          | 0/26 [00:00<?, ?it/s]

    [02] train 0.9901 | val 0.8630 | gap 0.1271


...

    [05] train 1.0000 | val 0.8848 | gap 0.1152


  0%|          | 0/26 [00:00<?, ?it/s]

    [06] train 1.0000 | val 0.8810 | gap 0.1190


  0%|          | 0/26 [00:00<?, ?it/s]

    [07] train 1.0000 | val 0.8773 | gap 0.1227
    Early stop. Best val 0.8922 @ epoch 2

── Stage 2 Summary ──────────────────────────────────────────────────────
Config                          mean     std      se   med_B
----------------------------------------------------------
B/16 unfreeze_2_1e5           0.8619  0.0145  0.0073       2
L/14 unfreeze_2_1e5           0.8925  0.0069  0.0035       2

Decision threshold for Stage 3: B/16 mean + 2×SE = 0.8619 + 2×0.0073 = 0.8765
L/14 unfreeze PASSES ✓ (proceed → flip RUN_L14_SUBMISSION)


In [11]:
# ── Stage 3: ViT-L/14 fold-ensemble submission (GATED) ───────────────────
# Flip RUN_L14_SUBMISSION = True only if Stage 2 L/14 mean exceeds B/16
# unfreeze_2_1e5 mean by ~2 SE: l14_uf_mean > 0.8619 + 2×0.0073 ≈ 0.8765
RUN_L14_SUBMISSION = True

if not RUN_L14_SUBMISSION:
    print('GATED — set RUN_L14_SUBMISSION = True to run Stage 3.')
    print(f'Proceed only if Stage 2 L/14 mean > 0.8619 + 2×0.0073 = {0.8619+2*0.0073:.4f}')
else:
    _l14_ckpt_paths = sorted(glob.glob(
        os.path.join(CKPT_DIR, 'l14_unfreeze_2_1e5_fold*_best.pt')
    ))
    assert len(_l14_ckpt_paths) == 4, \
        f'Expected 4 L/14 checkpoints, found {len(_l14_ckpt_paths)}: {_l14_ckpt_paths}'
    print(f'Found {len(_l14_ckpt_paths)} L/14 checkpoints:')
    for _p in _l14_ckpt_paths:
        print(f'  {os.path.basename(_p)}')

    _l14_test_ds     = TestDataset(TEST_DIR, transform=l14_preprocess)
    _l14_test_loader = DataLoader(_l14_test_ds, batch_size=L14_BATCH_SIZE,
                                   shuffle=False, num_workers=NUM_WORKERS)

    _l14_all_fold_logits = []
    _l14_all_fold_preds  = []

    for _ckpt_path in _l14_ckpt_paths:
        # Reset backbone — clf.clip IS l14_model; loading state_dict mutates it in-place
        l14_model.load_state_dict({k: v.to(device) for k, v in l14_original_state.items()})

        _ckpt = torch.load(_ckpt_path, map_location=device)
        _head_state = {k[len('head.'):]: v for k, v in _ckpt['model_state_dict'].items()
                       if k.startswith('head.')}
        _clf = make_l14_clf(_head_state, 2)
        _clf.load_state_dict({k: v.to(device) for k, v in _ckpt['model_state_dict'].items()})
        _clf.eval()

        _fold_logits = []
        with torch.no_grad():
            for _imgs, _ in tqdm(_l14_test_loader,
                                  desc=f'  {os.path.basename(_ckpt_path)}', leave=False):
                _imgs = _imgs.to(device)
                _fold_logits.append(_clf(_imgs).cpu())

        _fold_tensor = torch.cat(_fold_logits, dim=0)
        _l14_all_fold_logits.append(_fold_tensor)
        _l14_all_fold_preds.append(_fold_tensor.argmax(dim=1))

        del _clf, _fold_logits, _fold_tensor
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        elif torch.backends.mps.is_available(): torch.mps.empty_cache()

    _l14_ensemble_preds = (torch.stack(_l14_all_fold_logits, dim=0)
                            .sum(dim=0).argmax(dim=1).tolist())
    _l14_ensemble_ids   = [os.path.basename(_p) for _p in _l14_test_ds.paths]
    _sub_l14 = pd.DataFrame({'ID': _l14_ensemble_ids, 'Label': _l14_ensemble_preds})
    _sub_l14.to_csv('submission_l14_ensemble.csv', index=False)

    print(f'\nEnsembled {len(_l14_ckpt_paths)} L/14 models | {len(_sub_l14)} rows')
    print(_sub_l14.head(10))

    print('\nAgreement rates (L/14 ensemble vs each fold):')
    _l14_ens_arr = np.array(_l14_ensemble_preds)
    for _i, _fp in enumerate(_l14_all_fold_preds):
        print(f'  fold{_i}: {(_l14_ens_arr == _fp.numpy()).mean():.4f}')

    print('\nSaved: submission_l14_ensemble.csv')


Found 4 L/14 checkpoints:
  l14_unfreeze_2_1e5_fold0_best.pt
  l14_unfreeze_2_1e5_fold1_best.pt
  l14_unfreeze_2_1e5_fold2_best.pt
  l14_unfreeze_2_1e5_fold3_best.pt
    Unfrozen blocks: [22, 23] of 24


  l14_unfreeze_2_1e5_fold0_best.pt:   0%|          | 0/33 [00:00<?, ?it/s]

    Unfrozen blocks: [22, 23] of 24


  l14_unfreeze_2_1e5_fold1_best.pt:   0%|          | 0/33 [00:00<?, ?it/s]

    Unfrozen blocks: [22, 23] of 24


  l14_unfreeze_2_1e5_fold2_best.pt:   0%|          | 0/33 [00:00<?, ?it/s]

    Unfrozen blocks: [22, 23] of 24


  l14_unfreeze_2_1e5_fold3_best.pt:   0%|          | 0/33 [00:00<?, ?it/s]


Ensembled 4 L/14 models | 1036 rows
      ID  Label
0  0.jpg     62
1  1.jpg     43
2  2.jpg     38
3  3.jpg     51
4  4.jpg     42
5  5.jpg     89
6  6.jpg      3
7  7.jpg     28
8  8.jpg     71
9  9.jpg     71

Agreement rates (L/14 ensemble vs each fold):
  fold0: 0.9469
  fold1: 0.9315
  fold2: 0.9324
  fold3: 0.9498

Saved: submission_l14_ensemble.csv
